# CrewAI Demos · PaySetu's Failed-Payment Complaint Desk

> **This notebook is CrewAI only** (CrewAI Demo 1 and CrewAI Demo 2 in the slides). The AutoGen demos are in `Lesson_01_AutoGen_All_Demos.ipynb`.
> Every code cell starts with its tag so you can't mix them up:
> - `# ⬜ CREW SETUP` → run once at the start
> - `# 🟩 CREW DEMO 1` → the Payment Complaint Desk (receptionist + specialists)
> - `# 🟧 CREW DEMO 2` → One by one vs. all at once (+ a guardrail)

### The story (read this out)
Same company as the AutoGen demos: **PaySetu**, a UPI payments app in Mumbai. In AutoGen Demo 2, the audit found PaySetu's refund SOP breaks an RBI rule. Now we build the complaint desk **the right way**, with a CrewAI team.

**The real rule we use** (RBI circular on turnaround time for failed transactions, 20 September 2019): if a UPI payment fails, the customer is debited but the merchant is not credited, the money must be **auto-reversed by the next day (T+1)**. After that, the customer gets **₹100 for every day of delay**, without having to ask.

| Customer | Transaction | What really happened | What the team should do |
|---|---|---|---|
| Rahul | UPI-4417, ₹2,499 at Reliance Digital, 10 Oct | Failed, debited, **still not reversed** | Refund ₹2,499 + ₹600 compensation (6 days late) |
| Priya | UPI-4418, ₹1,180 electricity bill, 14 Oct | Failed, reversed on 15 Oct (on time) | No compensation; show the reversal |
| Arjun | UPI-4419, ₹640 grocery order, 16 Oct | **Actually succeeded**: the merchant got the money | No refund from PaySetu; ask the merchant |

---
# ⬜ CREW SETUP (run once, before either demo)

## Setup 1 · Install (run once, before class)

In [1]:
# ⬜ CREW SETUP 1 · Install (run once)
%pip install -q -U crewai langchain-openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


## Setup 2 · Connect to gpt-5-mini (same gateway as every other notebook)
The first three lines switch off CrewAI's anonymous usage reporting and tracing prompt, to keep the output clean and avoid outside calls the lab network may block.

In [2]:
# ⬜ CREW SETUP 2 · Connect to gpt-5-mini through the Azure gateway
import os
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["CREWAI_TRACING_ENABLED"] = "false"

from dotenv import load_dotenv, find_dotenv
from langchain_openai import AzureChatOpenAI

load_dotenv(find_dotenv(usecwd=True), override=True)
api_key="2ABecnfxzhRg4M5D6pBKiqxXVhmGB2WvQ0aYKkbTCPsj0JLKsZPfJQQJ99BDAC77bzfXJ3w3AAABACOGi3sC"
GATEWAY = os.getenv("AZURE_GATEWAY_URL", "https://openai-api-management-gw.azure-api.net")

llm = AzureChatOpenAI(
    base_url=f"{GATEWAY}/openaiprodtest-clone",
    model="gpt-5-mini",
    api_version="2024-12-01-preview",
    api_key=api_key,
    validate_base_url=False,
)
# No temperature on purpose: gpt-5-mini only accepts the default.
print("Model says:", llm.invoke("Reply with just the word: ready").content)

Model says: ready


## Setup 3 · Adapter so CrewAI can use our gateway
CrewAI normally connects to models its own way, and gpt-5-mini rejects two things it usually sends (a `temperature`, and "stop words"). This small adapter sends every message through the **same `llm` connection that already works**, and does the "stop" step itself.

**Explain in class (one line):** *"A travel adapter plug. Same electricity, different socket."*

In [3]:
#  CREW SETUP 3 · Adapter: lets CrewAI use the working gateway connection
from crewai import BaseLLM

class GatewayLLM(BaseLLM):
    def call(self, messages, tools=None, callbacks=None, available_functions=None, **kwargs):
        if isinstance(messages, str):
            messages = [{"role": "user", "content": messages}]
        chat = [{"role": m.get("role", "user"), "content": str(m.get("content", ""))} for m in messages]
        text = llm.invoke(chat).content
        if "\nObservation:" in text:          # stop before the model invents a tool result
            text = text.split("\nObservation:")[0]
        return text

    def supports_function_calling(self):
        return False      # agents use tools by writing "Action: ... / Action Input: ..."

    def supports_stop_words(self):
        return False      # handled inside call() above

    def get_context_window_size(self):
        return 128000

crew_llm = GatewayLLM(model="gpt-5-mini")
print("CrewAI adapter says:", crew_llm.call("Reply with just the word: ready"))

CrewAI adapter says: ready


## Setup 4 · Start a crew safely inside Jupyter
Newer CrewAI versions refuse to start a crew inside Jupyter's background "event loop". This helper starts it in a separate worker thread. **In class just say:** *"`run_crew` means: start the team working."*

In [4]:
# ⬜ CREW SETUP 4 · Helper: start a crew safely inside Jupyter
from concurrent.futures import ThreadPoolExecutor

def run_crew(crew, inputs):
    with ThreadPoolExecutor(max_workers=1) as worker:
        return worker.submit(crew.kickoff, inputs=inputs).result()

## Setup 5 · PaySetu's data: transactions, the RBI rule, the help handbook
**What it does:** three real-looking UPI transactions (the table at the top), the RBI rule, and PaySetu's help handbook.

**Point out:** the compensation is calculated by **plain code**, not by the AI. *"We never let AI decide how much money to pay. It reads, explains and writes; the rules calculate."*

In [5]:
# ⬜ CREW SETUP 5 · Transactions, the RBI rule, the compensation calculator, the help handbook
from datetime import date, timedelta

TODAY = date(2026, 10, 17)

TRANSACTIONS = {
    "UPI-4417": {"customer": "Rahul Mehta", "paid_to": "Reliance Digital, Andheri", "amount": 2499,
                 "date": "2026-10-10", "status": "FAILED", "customer_debited": True,
                 "merchant_credited": False, "reversed_on": None},
    "UPI-4418": {"customer": "Priya Nair", "paid_to": "MSEDCL electricity bill", "amount": 1180,
                 "date": "2026-10-14", "status": "FAILED", "customer_debited": True,
                 "merchant_credited": False, "reversed_on": "2026-10-15"},
    "UPI-4419": {"customer": "Arjun Shah", "paid_to": "Zepto grocery order", "amount": 640,
                 "date": "2026-10-16", "status": "SUCCESS", "customer_debited": True,
                 "merchant_credited": True, "reversed_on": None},
}

RBI_RULE = (
    "RBI circular on turnaround time (TAT) for failed transactions, 20 Sept 2019: for UPI, if the customer's "
    "account is debited but the beneficiary is not credited, the amount must be auto-reversed by T+1 day "
    "(the day after the transaction). If reversal is later than that, the customer must be paid Rs 100 per day "
    "of delay, automatically, without needing to complain. If the beneficiary WAS credited, the bank does not "
    "refund; the customer should contact the merchant."
)

def compensation_for(txn_id):
    """Plain rules, no AI: refund due and RBI compensation as of TODAY."""
    t = TRANSACTIONS.get(txn_id.strip().upper())
    if not t:
        return None
    if t["merchant_credited"] or not t["customer_debited"]:
        return {"refund_due": 0, "days_late": 0, "compensation": 0, "note": "Merchant was paid. No refund from PaySetu."}
    due_by = date.fromisoformat(t["date"]) + timedelta(days=1)             # T+1
    returned = date.fromisoformat(t["reversed_on"]) if t["reversed_on"] else TODAY
    days_late = max(0, (returned - due_by).days)
    return {"refund_due": 0 if t["reversed_on"] else t["amount"], "due_by": str(due_by),
            "days_late": days_late, "compensation": 100 * days_late}

HELP_HANDBOOK = """
- UPI daily limit: Rs 1,00,000 for most payments (NPCI limit). Up to Rs 5,00,000 per transaction for verified hospitals and educational institutions.
- Changing your mobile number: link the new number to your bank account at the bank first, then re-register UPI in the PaySetu app.
- Customer care: 24x7 in the app chat; phone line 8 AM to 10 PM.
- Unhappy with our answer? If a complaint is not resolved within 30 days, you can complain to the RBI Ombudsman at cms.rbi.org.in.
- Never share your UPI PIN or OTP. PaySetu staff will never ask for it.
"""

print("Compensation check for UPI-4417:", compensation_for("UPI-4417"))

Compensation check for UPI-4417: {'refund_due': 2499, 'due_by': '2026-10-11', 'days_late': 6, 'compensation': 600}


## Setup 6 · The tools (the team's equipment)
| Tool | Who gets it | What it does |
|---|---|---|
| `look_up_transaction` | Disputes Specialist, Investigator | Finds a transaction by ID |
| `check_rbi_rule` | Disputes Specialist, Rules Expert | Returns the RBI rule text |
| `calculate_compensation` | Disputes Specialist, Reply Writer | Runs the plain-code calculator |
| `read_help_handbook` | Help Desk Assistant | Reads the FAQ handbook |

**Point out:** the sentence under each tool is **how the agent knows when to use it**, like the label on a tool in a toolbox.

In [6]:
# ⬜ CREW SETUP 6 · Tools = the equipment we hand to agents
import json
from crewai.tools import tool

@tool("look_up_transaction")
def look_up_transaction(txn_id: str) -> str:
    """Finds a PaySetu UPI transaction by its ID (like UPI-4417): customer, merchant, amount, date, status, whether the customer was debited, whether the merchant was credited, and any reversal date."""
    t = TRANSACTIONS.get(txn_id.strip().upper())
    return json.dumps({"txn_id": txn_id.upper(), **t}) if t else f"No transaction found with ID {txn_id}."

@tool("check_rbi_rule")
def check_rbi_rule(topic: str) -> str:
    """Returns the official RBI rule for failed UPI transactions: reversal deadline and compensation."""
    return RBI_RULE

@tool("calculate_compensation")
def calculate_compensation(txn_id: str) -> str:
    """Calculates, for a UPI transaction ID, the refund still due and the RBI compensation owed as of today. Always use this instead of calculating yourself."""
    result = compensation_for(txn_id)
    return json.dumps(result) if result else f"No transaction found with ID {txn_id}."

@tool("read_help_handbook")
def read_help_handbook(topic: str) -> str:
    """Returns PaySetu's help handbook: UPI limits, changing mobile number, customer care hours, the RBI Ombudsman, and safety tips."""
    return HELP_HANDBOOK

print("Tools ready ✅")

Tools ready ✅


---
# 🟩 CREW DEMO 1 · The Payment Complaint Desk (routing to the right desk)

**Say before starting:** *"Every bank app has this problem: thousands of messages, very different kinds. A receptionist sorts them; specialists answer."*

```
Customer message ─► 🧑‍💼 Receptionist ──"DISPUTE"──► 💳 Payment Disputes Specialist (transaction, RBI rule, calculator)
                                     └─"GENERAL"─► 📘 Help Desk Assistant (help handbook)
```

## Crew Demo 1 · Step 1 · Hire the team
**What it does:** three agents, each with a **role** (job title), **goal** (what success looks like) and **backstory** (experience and personality).

**Explain in class:** read the Disputes Specialist aloud. *"It reads like a job description. That's the big idea of CrewAI."*

**Point out:** the Receptionist gets **no tools**: it only sorts, it never touches money. `allow_delegation=False` keeps each agent in its own lane. `verbose=True` shows the agents thinking on screen.

In [7]:
# 🟩 CREW DEMO 1 · Step 1 · Agents: Receptionist, Payment Disputes Specialist, Help Desk Assistant
from crewai import Agent

receptionist = Agent(
    role="Customer Care Receptionist",
    goal="Read each customer message and send it to the right desk",
    backstory="You have sorted thousands of PaySetu messages. You are quick and you never try to solve the problem yourself.",
    llm=crew_llm, allow_delegation=False, verbose=True,
)

disputes_specialist = Agent(
    role="Payment Disputes Specialist",
    goal="Resolve failed-payment complaints exactly as the RBI rule requires",
    backstory=("You always look up the transaction first, check the RBI rule, and use the calculator for any amount. "
               "You never promise money the rules don't allow, and you never hold back money the rules require."),
    tools=[look_up_transaction, check_rbi_rule, calculate_compensation],
    llm=crew_llm, allow_delegation=False, verbose=True,
)

help_desk = Agent(
    role="Help Desk Assistant",
    goal="Answer general questions using only PaySetu's help handbook",
    backstory="You are friendly and brief. If the handbook doesn't say it, you say the team will check and reply.",
    tools=[read_help_handbook],
    llm=crew_llm, allow_delegation=False, verbose=True,
)
print("Team hired ✅")

Team hired ✅


## Crew Demo 1 · Step 2 · Give them work (tasks)
**What it does:** a **task** is one piece of work for one agent: a description, plus `expected_output` (what "done" looks like). `{message}` is a blank filled in with the customer's message.

In [8]:
# 🟩 CREW DEMO 1 · Step 2 · Tasks: sort, resolve a dispute, answer a question
from crewai import Task

sort_task = Task(
    description=("Customer message: {message}\n\n"
                 "Decide which desk should handle it:\n"
                 "DISPUTE = a payment failed, money was debited, a refund or reversal is asked for\n"
                 "GENERAL = questions about limits, mobile number, timings, complaints process, safety\n"
                 "Answer with exactly one word: DISPUTE or GENERAL."),
    expected_output="Exactly one word: DISPUTE or GENERAL",
    agent=receptionist,
)

dispute_task = Task(
    description=("Customer message: {message}\n\n"
                 "Look up the transaction, check the RBI rule, and use the calculator. Then write a reply to the "
                 "customer in under 100 words: what happened, what PaySetu will pay and by when, and the RBI "
                 "Ombudsman option if they are not satisfied."),
    expected_output="A short, polite, accurate reply with the transaction ID and any refund and compensation amounts",
    agent=disputes_specialist,
)

help_task = Task(
    description="Customer message: {message}\n\nAnswer using the help handbook, in under 70 words.",
    expected_output="A short, friendly answer based only on the handbook",
    agent=help_desk,
)
print("Tasks ready ✅")

Tasks ready ✅


## Crew Demo 1 · Step 3 · The routing helper
**What it does:** runs the Receptionist alone, reads its one-word decision, then runs the right specialist. A **Crew** is a team plus their tasks.

**Explain in class:** *"The receptionist decides; the code just follows the decision, like a token system sending you to counter 4."*

In [9]:
# 🟩 CREW DEMO 1 · Step 3 · Helper: Receptionist decides, then the right desk answers
from crewai import Crew

def handle_message(message):
    print("📩 Customer:", message, "\n")
    decision = run_crew(Crew(agents=[receptionist], tasks=[sort_task]), {"message": message}).raw.strip().upper()
    desk = "DISPUTE" if "DISPUTE" in decision else "GENERAL"
    print(f"\n🧑‍💼 Receptionist decided: {desk}\n")

    if desk == "DISPUTE":
        crew = Crew(agents=[disputes_specialist], tasks=[dispute_task])
    else:
        crew = Crew(agents=[help_desk], tasks=[help_task])
    answer = run_crew(crew, {"message": message}).raw
    print("\n" + "=" * 70 + f"\n💬 Reply from the {desk} desk:\n{answer}\n" + "=" * 70)

## Crew Demo 1 · Step 4 · Run: Rahul's failed payment
**Say before running:** *"Rahul paid ₹2,499 at Reliance Digital a week ago. Failed, debited, not back. Which desk?"*

**Point out while it runs:**
- The Receptionist says DISPUTE and uses **no tools**.
- The Specialist writes `Action: look_up_transaction` → result → `check_rbi_rule` → `calculate_compensation`. That's plan → act → check.
- The **₹600** (6 days late × ₹100) comes from the calculator, not the AI's imagination. Most customers don't know this rule exists.

In [10]:
# 🟩 CREW DEMO 1 · Step 4 · Run: a failed payment that was never reversed
handle_message("I paid Rs 2,499 at Reliance Digital on 10 Oct (UPI-4417). Payment failed but money was debited and it's still not back!")

📩 Customer: I paid Rs 2,499 at Reliance Digital on 10 Oct (UPI-4417). Payment failed but money was debited and it's still not back! 



╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Care Receptionist                                                                              │
│                                                                                                                 │
│  Task: Customer message: I paid Rs 2,499 at Reliance Digital on 10 Oct (UPI-4417). Payment failed but money     │
│  was debited and it's still not back!                                                                           │
│                                                                                                                 │
│  Decide which desk should handle it:                                                                            │
│  DISPUTE = a payment failed, money was debited, a refund or reversal is asked for                               │
│  GENERAL = questions about limits, mobile number, timings, complaints process, safety                           │
│  Answer with exactly one word: DISPUTE or GENERAL.                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Care Receptionist                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  DISPUTE                                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


🧑‍💼 Receptionist decided: DISPUTE



...

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  RBI circular on turnaround time (TAT) for failed transactions, 20 Sept 2019: for UPI, if the customer's        │
│  account is debited but the beneficiary is not credited, the amount must be auto-reversed by T+1 day (the day   │
│  after the transaction). If reversal is later than that, the customer must be paid Rs 100 per day of delay,     │
│  automatically, without needing to complain. If the beneficiary WAS credited, the bank does not refund; the     │
│  customer should contact the merchant.                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  {"refund_due": 2499, "due_by": "2026-10-11", "days_late": 6, "compensation": 600}                              │
│                                                                                                                 │
│                                                                                                                 │
│  You ONLY have access to the following tools, and should NEVER make up tools that are not listed here:          │
│                                                                                                                 │
│  Tool Name: look_up_transaction                                                                                 │
│  Tool Arguments: {                                                                                              │
│    "properties": {                                                                                              │
│      "txn_id": {                                                                                                │
│        "title": "Txn Id",                                                                                       │
│        "type": "string"                                                                                         │
│      }                                                                                                          │
│    },                                                                                                           │
│    "required": [                                                                                                │
│      "txn_id"                                                                                                   │
│    ],                                                                                                           │
│    "title": "Look_Up_Transaction",                                                                              │
│    "type": "object",                                                                                            │
│    "additionalProperties": false                                                                                │
│  }                                                                                                              │
│  Tool Description: Finds a PaySetu UPI transaction by its ID (like UPI-4417): customer, merchant, amount,       │
│  date, status, whether the customer was debited, whether the merchant was credited, and any reversal date.      │
│  Tool Name: check_rbi_rule                                                                                      │
│  Tool Arguments: {                                                                                              │
│    "properties": {                                                                                              │
│      "topic": {                                                                                                 │
│        "title": "Topic",                                                                                        │
│        "type": "string"                                                                                         │
│      }                                                                                                          │
│    },                                                                                                           │
│    "required": [                                                                                                │
│      "topic"                                                                                                    │
│    ],                                                 

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Payment Disputes Specialist                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Transaction UPI-4417 (10 Oct) failed: your account was debited (Rs 2,499) but Reliance Digital was not         │
│  credited, and reversal is overdue. PaySetu will refund Rs 2,499 plus RBI compensation Rs 600 (total Rs 3,099)  │
│  — we will process this and it should reflect in your account within 2 working days. If you remain              │
│  unsatisfied, you may approach the RBI Ombudsman.                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


💬 Reply from the DISPUTE desk:
Transaction UPI-4417 (10 Oct) failed: your account was debited (Rs 2,499) but Reliance Digital was not credited, and reversal is overdue. PaySetu will refund Rs 2,499 plus RBI compensation Rs 600 (total Rs 3,099) — we will process this and it should reflect in your account within 2 working days. If you remain unsatisfied, you may approach the RBI Ombudsman.


## Crew Demo 1 · Step 5 · Run: a general question
**Point out:** the Receptionist picks GENERAL, and a different agent with a different tool answers. The Disputes Specialist never wakes up, so no time or money is wasted.

In [11]:
# 🟩 CREW DEMO 1 · Step 5 · Run: general question
handle_message("I need to pay my son's college fees of Rs 2 lakh by UPI. Is there a limit?")

📩 Customer: I need to pay my son's college fees of Rs 2 lakh by UPI. Is there a limit? 



╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Care Receptionist                                                                              │
│                                                                                                                 │
│  Task: Customer message: I need to pay my son's college fees of Rs 2 lakh by UPI. Is there a limit?             │
│                                                                                                                 │
│  Decide which desk should handle it:                                                                            │
│  DISPUTE = a payment failed, money was debited, a refund or reversal is asked for                               │
│  GENERAL = questions about limits, mobile number, timings, complaints process, safety                           │
│  Answer with exactly one word: DISPUTE or GENERAL.                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Care Receptionist                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  GENERAL                                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


🧑‍💼 Receptionist decided: GENERAL



╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Help Desk Assistant                                                                                     │
│                                                                                                                 │
│  Task: Customer message: I need to pay my son's college fees of Rs 2 lakh by UPI. Is there a limit?             │
│                                                                                                                 │
│  Answer using the help handbook, in under 70 words.                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│  - UPI daily limit: Rs 1,00,000 for most payments (NPCI limit). Up to Rs 5,00,000 per transaction for verified  │
│  hospitals and educational institutions.                                                                        │
│  - Changing your mobile number: link the new number to your bank account at the bank first, then re-register    │
│  UPI in the PaySetu app.                                                                                        │
│  - Customer care: 24x7 in the app chat; phone line 8 AM to 10 PM.                                               │
│  - Unhappy with our answer? If a complaint is not resolved within 30 days, you can complain to the RBI          │
│  Ombudsman at cms.rbi.org.in.                                                                                   │
│  - Never share your UPI PIN or OTP. PaySetu staff will never ask for it.                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Help Desk Assistant                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  UPI daily limit is Rs 1,00,000 for most payments (NPCI limit). Up to Rs 5,00,000 per transaction for verified  │
│  hospitals and educational institutions—so Rs 2,00,000 will work if the college is a verified educational       │
│  institution; otherwise you’ll hit the Rs 1,00,000 limit.                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


💬 Reply from the GENERAL desk:
UPI daily limit is Rs 1,00,000 for most payments (NPCI limit). Up to Rs 5,00,000 per transaction for verified hospitals and educational institutions—so Rs 2,00,000 will work if the college is a verified educational institution; otherwise you’ll hit the Rs 1,00,000 limit.


## Crew Demo 1 · Step 6 · Your turn: the tricky one
Arjun's grocery payment **shows failed in his app**, but the merchant actually received the money.

**Ask before running:** *"Should PaySetu refund him?"* (Many say yes.) Then run it. The Specialist looks it up, sees the merchant was credited, and correctly says: **contact the merchant**. A human agent in a hurry might have refunded twice.

Other messages to try from the room: Priya's `UPI-4418`, "How do I change my mobile number?", "Someone called asking for my UPI PIN".

In [12]:
# 🟩 CREW DEMO 1 · Step 6 · Your turn
handle_message("My Zepto payment UPI-4419 for Rs 640 shows failed in my app but the money is gone. Refund please!")

📩 Customer: My Zepto payment UPI-4419 for Rs 640 shows failed in my app but the money is gone. Refund please! 



╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Care Receptionist                                                                              │
│                                                                                                                 │
│  Task: Customer message: My Zepto payment UPI-4419 for Rs 640 shows failed in my app but the money is gone.     │
│  Refund please!                                                                                                 │
│                                                                                                                 │
│  Decide which desk should handle it:                                                                            │
│  DISPUTE = a payment failed, money was debited, a refund or reversal is asked for                               │
│  GENERAL = questions about limits, mobile number, timings, complaints process, safety                           │
│  Answer with exactly one word: DISPUTE or GENERAL.                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Care Receptionist                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  DISPUTE                                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


🧑‍💼 Receptionist decided: DISPUTE



...

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  RBI circular on turnaround time (TAT) for failed transactions, 20 Sept 2019: for UPI, if the customer's        │
│  account is debited but the beneficiary is not credited, the amount must be auto-reversed by T+1 day (the day   │
│  after the transaction). If reversal is later than that, the customer must be paid Rs 100 per day of delay,     │
│  automatically, without needing to complain. If the beneficiary WAS credited, the bank does not refund; the     │
│  customer should contact the merchant.                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  {"refund_due": 0, "days_late": 0, "compensation": 0, "note": "Merchant was paid. No refund from PaySetu."}     │
│                                                                                                                 │
│                                                                                                                 │
│  You ONLY have access to the following tools, and should NEVER make up tools that are not listed here:          │
│                                                                                                                 │
│  Tool Name: look_up_transaction                                                                                 │
│  Tool Arguments: {                                                                                              │
│    "properties": {                                                                                              │
│      "txn_id": {                                                                                                │
│        "title": "Txn Id",                                                                                       │
│        "type": "string"                                                                                         │
│      }                                                                                                          │
│    },                                                                                                           │
│    "required": [                                                                                                │
│      "txn_id"                                                                                                   │
│    ],                                                                                                           │
│    "title": "Look_Up_Transaction",                                                                              │
│    "type": "object",                                                                                            │
│    "additionalProperties": false                                                                                │
│  }                                                                                                              │
│  Tool Description: Finds a PaySetu UPI transaction by its ID (like UPI-4417): customer, merchant, amount,       │
│  date, status, whether the customer was debited, whether the merchant was credited, and any reversal date.      │
│  Tool Name: check_rbi_rule                                                                                      │
│  Tool Arguments: {                                                                                              │
│    "properties": {                                                                                              │
│      "topic": {                                                                                                 │
│        "title": "Topic",                                                                                        │
│        "type": "string"                                                                                         │
│      }                                                                                                          │
│    },                                                                                                           │
│    "required": [                                                                                                │
│      "topic"                                                                                                    │
│    ],                                                 

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Payment Disputes Specialist                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Hi Arjun — UPI-4419 for ₹640 succeeded: your account was debited and Zepto was credited, so PaySetu cannot     │
│  auto-reverse. Refund due: ₹0; RBI compensation: ₹0. Please contact Zepto for a refund. If unresolved, you may  │
│  file a complaint with the Banking Ombudsman (RBI).                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


💬 Reply from the DISPUTE desk:
Hi Arjun — UPI-4419 for ₹640 succeeded: your account was debited and Zepto was credited, so PaySetu cannot auto-reverse. Refund due: ₹0; RBI compensation: ₹0. Please contact Zepto for a refund. If unresolved, you may file a complaint with the Banking Ombudsman (RBI).


### 🟩 Crew Demo 1 wrap-up (say this)
*"Three agents, three job descriptions, and each one only had the tools for its job. The money was calculated by rules, not by AI. And the tricky case shows why agents must look things up instead of guessing."*

---
# 🟧 CREW DEMO 2 · One by one vs. all at once (+ a guardrail)

**Say before starting:** *"To reply to Rahul properly we need two things: the transaction facts and the RBI rule. Do those two jobs need to wait for each other?"* (No.)

```
ONE BY ONE:   🔎 Investigator ──► 📜 RBI Rules Expert ──► ✍️ Reply Writer (+ calculator) ──► 🛡️ guardrail

ALL AT ONCE:  🔎 Investigator ─┐
                               ├──► ✍️ Reply Writer (+ calculator) ──► 🛡️ guardrail
              📜 Rules Expert ─┘
```

## Crew Demo 2 · Step 1 · Hire the second team
- 🔎 **Transaction Investigator**: finds the facts (tool: `look_up_transaction`)
- 📜 **RBI Rules Expert**: states the rule (tool: `check_rbi_rule`)
- ✍️ **Reply Writer**: writes the reply, and uses the calculator for the amounts (tool: `calculate_compensation`)

`verbose=False` this time, so we can focus on **time taken**.

In [13]:
# 🟧 CREW DEMO 2 · Step 1 · Agents: Investigator, RBI Rules Expert, Reply Writer
investigator = Agent(
    role="Transaction Investigator",
    goal="Report the exact facts of the customer's transaction",
    backstory="You never assume. You always check the transaction system and report facts with dates and amounts.",
    tools=[look_up_transaction], llm=crew_llm, allow_delegation=False, verbose=False,
)
rules_expert = Agent(
    role="RBI Rules Expert",
    goal="State exactly which RBI rule applies and what it requires",
    backstory="You quote regulations precisely from the official rule tool and never from memory.",
    tools=[check_rbi_rule], llm=crew_llm, allow_delegation=False, verbose=False,
)
reply_writer = Agent(
    role="Customer Reply Writer",
    goal="Write warm, clear, accurate replies that customers trust",
    backstory="You write like a helpful bank manager: polite, honest and specific. You always use the calculator for amounts.",
    tools=[calculate_compensation], llm=crew_llm, allow_delegation=False, verbose=False,
)
print("Second team hired ✅")

Second team hired ✅


## Crew Demo 2 · Step 2 · A guardrail: check the reply before it goes out
**What it does:** a small Python check that runs **after** the Reply Writer finishes. If the reply doesn't mention the transaction ID **and** the correct compensation amount, it's sent back with the reason (up to 2 retries). This is the "quality checker at the factory gate" from the slides.

`return (True, output)` = pass. `return (False, "reason")` = send it back.

In [14]:
# 🟧 CREW DEMO 2 · Step 2 · Guardrail: the reply must mention the transaction ID and the right compensation
TXN = "UPI-4417"
COMPLAINT = "I paid Rs 2,499 at Reliance Digital on 10 Oct (UPI-4417). It failed, money was debited, and it's still not back after a week."

def reply_guardrail(output):
    text = output.raw.replace(",", "")
    expected = compensation_for(TXN)["compensation"]
    problems = []
    if TXN not in text:
        problems.append(f"mention the transaction ID {TXN}")
    if expected and str(expected) not in text:
        problems.append(f"state the compensation of Rs {expected}")
    if problems:
        print("🛡️ Guardrail: reply sent back →", "; ".join(problems))
        return (False, "Please " + " and ".join(problems) + ".")
    print("🛡️ Guardrail: reply passed ✅")
    return (True, output)

## Crew Demo 2 · Step 3 · The tasks, with one switch
- `async_execution=True` → *"don't wait for anyone, start now"*. With the switch on, the Investigator and the Rules Expert start together.
- `context=[...]` → the Reply Writer waits for both results. You can't write the reply before you have the facts.
- `guardrail=reply_guardrail` → the check from Step 2.

In [15]:
# 🟧 CREW DEMO 2 · Step 3 · Tasks, with a switch for one-by-one vs. all-at-once
def make_tasks(parallel):
    investigate = Task(
        description="Complaint: {complaint}\nLook up the transaction and report: customer, merchant, amount, date, status, debited/credited, reversal date.",
        expected_output="Bullet points of facts", agent=investigator, async_execution=parallel,
    )
    find_rule = Task(
        description="Complaint: {complaint}\nState the RBI rule that applies: the reversal deadline and the compensation.",
        expected_output="The rule in 2-3 plain sentences", agent=rules_expert, async_execution=parallel,
    )
    write = Task(
        description=("Using the facts and the rule, use the calculator for this transaction, then write a reply to the customer "
                     "in under 100 words: the transaction ID, the refund, the compensation amount, and the RBI Ombudsman option."),
        expected_output="A short, warm, accurate reply",
        agent=reply_writer, context=[investigate, find_rule],
        guardrail=reply_guardrail, guardrail_max_retries=2,
    )
    return [investigate, find_rule, write]

## Crew Demo 2 · Step 4 · Run ONE BY ONE (serial)
**Say:** *"First the queue: Investigator, then Rules Expert, then Writer."* Write the time on the whiteboard.

`run_team` runs the crew, prints each agent's output, and returns the time. If the reply fails the guardrail 3 times, CrewAI refuses to finish, and the cell says so instead of crashing.

In [16]:
# 🟧 CREW DEMO 2 · Step 4 · Run serial (one by one) and time it
import time
from crewai import Process

def run_team(parallel):
    start = time.perf_counter()
    try:
        result = run_crew(Crew(agents=[investigator, rules_expert, reply_writer],
                               tasks=make_tasks(parallel), process=Process.sequential),
                          {"complaint": COMPLAINT})
        for t in result.tasks_output:
            print(f"--- {t.agent} ---\n{t.raw}\n")
    except Exception as e:
        # If the reply fails the guardrail 3 times, CrewAI stops instead of sending it.
        print("🛡️ Blocked:", e, "\n→ The guardrail did its job: no wrong reply reached the customer.")
    return time.perf_counter() - start

serial_seconds = run_team(parallel=False)
print(f"⏱️  ONE BY ONE took {serial_seconds:.1f} seconds")

🛡️ Guardrail: reply sent back → mention the transaction ID UPI-4417
🛡️ Guardrail: reply passed ✅
--- Transaction Investigator ---
- Customer: Rahul Mehta
- Merchant: Reliance Digital, Andheri
- Amount: Rs 2,499
- Date: 2026-10-10
- Status: FAILED
- Customer debited: Yes
- Merchant credited: No
- Reversal date: No reversal recorded (reversed_on: null)

--- RBI Rules Expert ---
RBI circular on turnaround time (TAT) for failed transactions, 20 Sept 2019: for UPI, if the customer's account is debited but the beneficiary is not credited, the amount must be auto-reversed by T+1 day (the day after the transaction). If reversal is later than that, the customer must be paid Rs 100 per day of delay, automatically, without needing to complain. If the beneficiary WAS credited, the bank does not refund; the customer should contact the merchant.

--- Customer Reply Writer ---
Dear Rahul Mehta, Transaction ID UPI-4417: refund due Rs 2,499 and RBI compensation Rs 600 (6 days delay). If unresolved, you

## Crew Demo 2 · Step 5 · Run ALL AT ONCE (parallel)
**Say:** *"Same team, same complaint. Only the switch changes."*

**If it isn't faster:** a teaching moment, not a failure. The company gateway may process one request at a time. *"Parallel only helps if the road has more than one lane."*

In [17]:
# 🟧 CREW DEMO 2 · Step 5 · Run parallel (all at once) and time it
parallel_seconds = run_team(parallel=True)
print(f"⏱️  ALL AT ONCE took {parallel_seconds:.1f} seconds")

🛡️ Guardrail: reply passed ✅
--- Transaction Investigator ---
- Transaction ID: UPI-4417
- Customer: Rahul Mehta
- Merchant: Reliance Digital, Andheri
- Amount: Rs 2,499
- Date: 2026-10-10
- Status: FAILED
- Customer debited: Yes
- Merchant credited: No
- Reversal date: None (no reversal recorded)

--- RBI Rules Expert ---
RBI circular on turnaround time (TAT) for failed transactions, 20 Sept 2019: for UPI, if the customer's account is debited but the beneficiary is not credited, the amount must be auto-reversed by T+1 day (the day after the transaction). If reversal is later than that, the customer must be paid Rs 100 per day of delay, automatically, without needing to complain. If the beneficiary WAS credited, the bank does not refund; the customer should contact the merchant.

--- Customer Reply Writer ---
Dear Rahul Mehta—For UPI-4417 (Reliance Digital, Andheri) on 10‑Oct‑2026 the full refund due is Rs 2,499 and RBI compensation for delay is Rs 600 (6 days late). We will process th

## Crew Demo 2 · Step 6 · Compare
**Ask the room:**
1. *"Which was faster, and why?"*
2. *"When would all-at-once be a bad idea?"* → when one job needs another's result. The Writer always waits.
3. *"Did the guardrail ever send a reply back?"* If yes, point at the 🛡️ line. If no: *"the check still ran; it's the seatbelt you only notice when you need it."*

In [18]:
# 🟧 CREW DEMO 2 · Step 6 · Side-by-side comparison
print(f"One by one  (serial):   {serial_seconds:6.1f} s")
print(f"All at once (parallel): {parallel_seconds:6.1f} s")
saved = serial_seconds - parallel_seconds
print(f"Time saved: {saved:.1f} s" if saved > 0 else "No time saved this run (see the note above Step 5)")

One by one  (serial):    102.0 s
All at once (parallel):   62.1 s
Time saved: 39.8 s


### 🟧 Crew Demo 2 wrap-up (say this)
*"Same team, same work: one switch changed a queue into people working side by side. And a guardrail made sure no reply went out without the right numbers. That's how you'd want a real bank's complaint desk to work."*

---
## Link back to the slides
| Slide idea | Where it was |
|---|---|
| Roles / goals / backstory | Every `Agent(...)` |
| Tools per role | Receptionist: none. Each specialist: only its own tools |
| Serial vs. parallel | Crew Demo 2, Steps 4 and 5 (`async_execution=True`) |
| Guardrail | `reply_guardrail` (Crew Demo 2, Step 2) |
| AutoGen vs. CrewAI | AutoGen held a **meeting** to audit the policy; CrewAI ran a **process** to handle the complaints |

### If something goes wrong in class
| You see | Do this |
|---|---|
| Setup 2 or 3 fails | Check `.env` and the lab network. Fix before class. |
| `400` mentioning `stop` or `temperature` | Make sure every agent uses `llm=crew_llm` (the adapter). |
| `429` | Rate limit. Wait 30–60 seconds and re-run the cell. |
| `...invoked synchronously from within a running event loop` | Use `run_crew(...)`, not `crew.kickoff(...)`. |
| Guardrail fails 3 times | Re-run the cell; AI wording varies, the calculator's numbers don't. |